
# 12 — Thai Word-Level Dataset Preparation (V5)

Notebook นี้เตรียมข้อมูลจาก **sentence-level lip-reading dataset เดิม** ให้เป็น **word-level dataset** สำหรับ V5

แนวทาง:

```text
train/val/test sentence CSV
        ↓
Thai word segmentation
  AttaCut (ถ้ามี) / newmm fallback
        ↓
นับความถี่คำ
        ↓
เลือก vocabulary จาก TRAIN เท่านั้น
        ↓
สร้าง word occurrences
        ↓
[Optional] Faster-Whisper หา timed ASR tokens จากเสียง
        ↓
จับคู่ target word ↔ timed token แบบ conservative
        ↓
word_alignment_candidates.csv
```

> จุดสำคัญ: Notebook นี้ **ไม่หารเวลา subtitle เท่า ๆ กันตามจำนวนคำ**
> เพราะจะทำให้ word boundary ผิดและทำให้โมเดล lip-reading เรียน label ผิด

ผลลัพธ์หลักจะอยู่ที่:

```text
D:\Face_detection\dataset\word_level_v5\video001\
```

ไฟล์หลัก:
- `tokenized_sentences.csv`
- `word_frequency.csv`
- `selected_vocabulary.csv`
- `word_occurrences.csv`
- `asr_word_timestamps.csv` (ถ้ารัน Optional ASR)
- `word_alignment_candidates.csv` (ถ้ารัน Optional ASR)


## Cell 1 — Environment Check

In [ ]:

import sys
from pathlib import Path
import pandas as pd
import numpy as np

print("=" * 80)
print("ENVIRONMENT")
print("=" * 80)
print("Python :", sys.executable)
print("Version:", sys.version.split()[0])
print("Pandas :", pd.__version__)
print("NumPy  :", np.__version__)


## Cell 2 — Path / Config

In [ ]:

from pathlib import Path
import os

PROJECT_ROOT = Path(r"D:\Face_detection")
VIDEO_ID = "video001"

SPLIT_ROOT = PROJECT_ROOT / "dataset" / "splits" / VIDEO_ID

TRAIN_CSV = SPLIT_ROOT / "train.csv"
VAL_CSV   = SPLIT_ROOT / "val.csv"
TEST_CSV  = SPLIT_ROOT / "test.csv"

RAW_VIDEO = PROJECT_ROOT / "dataset" / "raw_videos" / f"{VIDEO_ID}.webm"

OUTPUT_ROOT = PROJECT_ROOT / "dataset" / "word_level_v5" / VIDEO_ID
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# เก็บ model/cache ที่ดาวน์โหลดใน D: ไม่ให้กิน C:
HF_HOME = PROJECT_ROOT / "cache" / "huggingface"
WHISPER_CACHE = PROJECT_ROOT / "cache" / "faster_whisper"
TMP_ROOT = PROJECT_ROOT / "temp" / "word_level_v5"

HF_HOME.mkdir(parents=True, exist_ok=True)
WHISPER_CACHE.mkdir(parents=True, exist_ok=True)
TMP_ROOT.mkdir(parents=True, exist_ok=True)

os.environ["HF_HOME"] = str(HF_HOME)
os.environ["HUGGINGFACE_HUB_CACHE"] = str(HF_HOME / "hub")
os.environ["TMP"] = str(TMP_ROOT)
os.environ["TEMP"] = str(TMP_ROOT)

# Word vocabulary config
MIN_TRAIN_COUNT = 10
MIN_VAL_COUNT   = 2
MIN_TEST_COUNT  = 2
MAX_VOCAB       = 100

# tokenizer: "attacut" หรือ "newmm"
TOKENIZER_ENGINE = "attacut"

print("PROJECT_ROOT :", PROJECT_ROOT)
print("OUTPUT_ROOT  :", OUTPUT_ROOT)
print("HF_HOME      :", HF_HOME)
print("RAW_VIDEO    :", RAW_VIDEO)

for p in [TRAIN_CSV, VAL_CSV, TEST_CSV]:
    print(p, "->", p.exists())
    if not p.exists():
        raise FileNotFoundError(p)



## Cell 3 — Install Thai Tokenizer (รันครั้งแรกเท่านั้น)

ถ้ายังไม่มี package ให้เอา `#` หน้า `%pip` ออกแล้วรัน

> เริ่มจาก `pythainlp + attacut` ก่อน  
> ยังไม่ต้องติดตั้ง Whisper ใน Cell นี้


In [ ]:

# รันครั้งแรกเท่านั้น ถ้า import ใน Cell ถัดไปไม่ผ่าน
# %pip install -U pythainlp attacut


## Cell 4 — Load Train / Val / Test

In [ ]:

def load_split(path, split_name):
    df = pd.read_csv(path)
    required = {
        "clip_id",
        "video_id",
        "segment_id",
        "speaker_id",
        "start_time",
        "end_time",
        "duration",
        "text",
    }
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{split_name}: missing columns = {missing}")

    df = df.copy()
    df["split"] = split_name
    df["speaker_id"] = df["speaker_id"].astype(int)
    df["segment_id"] = df["segment_id"].astype(int)
    df["start_time"] = df["start_time"].astype(float)
    df["end_time"] = df["end_time"].astype(float)
    df["duration"] = df["duration"].astype(float)
    df["text"] = df["text"].fillna("").astype(str).str.strip()
    return df

train_df = load_split(TRAIN_CSV, "train")
val_df   = load_split(VAL_CSV, "val")
test_df  = load_split(TEST_CSV, "test")

all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)

print("Train:", len(train_df))
print("Val  :", len(val_df))
print("Test :", len(test_df))
print("Total:", len(all_df))

display(all_df[
    ["clip_id", "speaker_id", "start_time", "end_time", "duration", "split", "text"]
].head(10))



## Cell 5 — Thai Word Segmentation

ลำดับการใช้:
1. พยายามใช้ **AttaCut**
2. ถ้าเครื่องยังไม่มี AttaCut จะ fallback ไป `newmm`

มีการ clean token:
- ตัด whitespace
- ไม่เก็บ punctuation ล้วน
- ไม่เก็บ token ว่าง


In [ ]:

import re

try:
    from pythainlp.tokenize import word_tokenize
except ImportError as e:
    raise ImportError(
        "ยังไม่มี pythainlp — กลับไปรัน Cell 3 ก่อน"
    ) from e

THAI_RE = re.compile(r"[\u0E00-\u0E7F]")
ALNUM_RE = re.compile(r"[A-Za-z0-9]")

def normalize_text(text):
    text = str(text).strip()
    text = re.sub(r"\s+", " ", text)
    return text

def clean_token(token):
    token = str(token).strip()
    token = re.sub(r"\s+", "", token)

    if not token:
        return None

    # เก็บ token ที่มีอักษรไทย หรือภาษาอังกฤษ/ตัวเลขอย่างน้อย 1 ตัว
    if not (THAI_RE.search(token) or ALNUM_RE.search(token)):
        return None

    # ลบ punctuation รอบนอก แต่ไม่ทำลายตัวอักษรไทย
    token = token.strip(".,!?;:'\"“”‘’()[]{}<>…-_+=/\\|@#$%^&*~`")
    token = token.strip()

    return token if token else None

def tokenize_thai(text, engine=TOKENIZER_ENGINE):
    text = normalize_text(text)

    try:
        tokens = word_tokenize(text, engine=engine, keep_whitespace=False)
        used_engine = engine
    except Exception as e:
        print(f"[WARN] {engine} ใช้ไม่ได้ -> fallback newmm | {type(e).__name__}: {e}")
        tokens = word_tokenize(text, engine="newmm", keep_whitespace=False)
        used_engine = "newmm"

    cleaned = []
    for t in tokens:
        t = clean_token(t)
        if t is not None:
            cleaned.append(t)

    return cleaned, used_engine

# quick test
samples = [
    "วันนี้เราจะไปโรงพยาบาลกันนะ",
    "ตอนนี้กำลังทดสอบระบบอ่านปากภาษาไทย",
]

for s in samples:
    tokens, engine_used = tokenize_thai(s)
    print("TEXT   :", s)
    print("ENGINE :", engine_used)
    print("TOKENS :", tokens)
    print()


## Cell 6 — Tokenize ทุกประโยค

In [ ]:

tokenized_rows = []
engine_counter = {}

for _, row in all_df.iterrows():
    tokens, engine_used = tokenize_thai(row["text"])
    engine_counter[engine_used] = engine_counter.get(engine_used, 0) + 1

    tokenized_rows.append({
        **row.to_dict(),
        "num_words": len(tokens),
        "tokens": "|".join(tokens),
        "tokenizer_engine": engine_used,
    })

tokenized_df = pd.DataFrame(tokenized_rows)

TOKENIZED_CSV = OUTPUT_ROOT / "tokenized_sentences.csv"
tokenized_df.to_csv(TOKENIZED_CSV, index=False, encoding="utf-8-sig")

print("Tokenizer usage:", engine_counter)
print("Saved:", TOKENIZED_CSV)
print()
print(tokenized_df["num_words"].describe())

display(tokenized_df[
    ["clip_id", "split", "speaker_id", "text", "num_words", "tokens"]
].head(20))



## Cell 7 — QA การตัดคำ

สุ่มดูประโยคจริงก่อนนับ vocabulary

ถ้าเห็นว่าตัดคำแปลกมาก ให้ **หยุดตรงนี้ก่อน** แล้วแก้ tokenizer/config


In [ ]:

sample_n = min(30, len(tokenized_df))

qa_sample = (
    tokenized_df
    .sample(n=sample_n, random_state=42)
    .sort_values(["split", "start_time"])
)

display(qa_sample[
    ["split", "clip_id", "text", "tokens", "num_words"]
])


## Cell 8 — Explode เป็น 1 แถวต่อ 1 คำ

In [ ]:

occurrence_rows = []

for _, row in tokenized_df.iterrows():
    tokens = [t for t in str(row["tokens"]).split("|") if t]

    for word_index, word in enumerate(tokens):
        occurrence_rows.append({
            "word_occurrence_id": f"{row['clip_id']}_w{word_index:03d}",
            "clip_id": row["clip_id"],
            "video_id": row["video_id"],
            "segment_id": int(row["segment_id"]),
            "speaker_id": int(row["speaker_id"]),
            "split": row["split"],
            "sentence_start": float(row["start_time"]),
            "sentence_end": float(row["end_time"]),
            "sentence_duration": float(row["duration"]),
            "sentence_text": row["text"],
            "word_index": int(word_index),
            "word": word,
            "num_words_in_sentence": len(tokens),
        })

word_occ_df = pd.DataFrame(occurrence_rows)

print("Total word occurrences:", len(word_occ_df))
print("Unique words          :", word_occ_df["word"].nunique())

display(word_occ_df.head(30))



## Cell 9 — Word Frequency

**สำคัญ:** เลือก vocabulary โดยดู `TRAIN` เป็นหลัก  
Val/Test ใช้ตรวจว่าคำนั้นมีตัวอย่างพอสำหรับประเมินหรือไม่


In [ ]:

freq = (
    word_occ_df
    .groupby(["word", "split"])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

for col in ["train", "val", "test"]:
    if col not in freq.columns:
        freq[col] = 0

freq["total"] = freq[["train", "val", "test"]].sum(axis=1)

freq = freq[
    ["word", "train", "val", "test", "total"]
].sort_values(
    ["train", "total", "word"],
    ascending=[False, False, True]
).reset_index(drop=True)

FREQ_CSV = OUTPUT_ROOT / "word_frequency.csv"
freq.to_csv(FREQ_CSV, index=False, encoding="utf-8-sig")

print("Saved:", FREQ_CSV)
display(freq.head(100))


## Cell 10 — เลือก Vocabulary สำหรับ V5

In [ ]:

selected_vocab = freq[
    (freq["train"] >= MIN_TRAIN_COUNT) &
    (freq["val"] >= MIN_VAL_COUNT) &
    (freq["test"] >= MIN_TEST_COUNT)
].copy()

selected_vocab = selected_vocab.head(MAX_VOCAB).reset_index(drop=True)
selected_vocab["class_id"] = np.arange(len(selected_vocab), dtype=int)

VOCAB_CSV = OUTPUT_ROOT / "selected_vocabulary.csv"
selected_vocab.to_csv(VOCAB_CSV, index=False, encoding="utf-8-sig")

print("=" * 80)
print("SELECTED VOCABULARY")
print("=" * 80)
print("MIN_TRAIN_COUNT:", MIN_TRAIN_COUNT)
print("MIN_VAL_COUNT  :", MIN_VAL_COUNT)
print("MIN_TEST_COUNT :", MIN_TEST_COUNT)
print("MAX_VOCAB      :", MAX_VOCAB)
print("Selected words :", len(selected_vocab))
print("Saved          :", VOCAB_CSV)

display(selected_vocab.head(100))



## Cell 11 — สร้าง Word Occurrence Manifest

ตอนนี้แต่ละคำรู้ว่าอยู่ใน **ประโยคไหน** และช่วงเวลา sentence เท่าไร  
แต่ **ยังไม่ถือว่า word timestamp ถูกต้อง**

จึงใช้:
- `word_start = NaN`
- `word_end = NaN`
- `alignment_status = pending`

ไว้ก่อน


In [ ]:

vocab_map = dict(zip(selected_vocab["word"], selected_vocab["class_id"]))

v5_occ = word_occ_df[word_occ_df["word"].isin(vocab_map)].copy()
v5_occ["class_id"] = v5_occ["word"].map(vocab_map).astype(int)

v5_occ["word_start"] = np.nan
v5_occ["word_end"] = np.nan
v5_occ["word_duration"] = np.nan
v5_occ["alignment_status"] = "pending"
v5_occ["alignment_confidence"] = np.nan
v5_occ["alignment_source"] = ""

WORD_OCC_CSV = OUTPUT_ROOT / "word_occurrences.csv"
v5_occ.to_csv(WORD_OCC_CSV, index=False, encoding="utf-8-sig")

print("Selected word occurrences:", len(v5_occ))
print("By split:")
print(v5_occ["split"].value_counts())
print("Saved:", WORD_OCC_CSV)

display(v5_occ.head(30))



# OPTIONAL — Word Timestamp ด้วย Faster-Whisper

ส่วนนี้ใช้ **เสียงเฉพาะตอนสร้าง dataset / alignment เท่านั้น**

โมเดล lip-reading V5 ตอน train/deploy จะใช้ **ภาพ Face + Mouth** ตามเดิม

ข้อควรทราบ:
- Whisper timestamp ไม่ควรถือว่าแม่น 100%
- ภาษาไทยไม่มีช่องว่างระหว่างทุกคำ
- เราจะเก็บเฉพาะ match ที่ conservative และส่งส่วนที่ไม่ชัดไป `review`
- ห้ามสร้าง word timestamp ด้วยการหารเวลาประโยคเท่า ๆ กัน



## Cell 12 — Install Faster-Whisper (Optional, รันครั้งแรกเท่านั้น)

> package/model cache ถูกตั้งไว้ที่ Drive D ใน Cell 2 แล้ว


In [ ]:

# เอา # ออกเฉพาะถ้าต้องการรัน ASR timestamp
%pip install -U faster-whisper


## Cell 13 — Check FFmpeg + Faster-Whisper

In [ ]:

import shutil

FFMPEG = shutil.which("ffmpeg")
print("ffmpeg:", FFMPEG)

if FFMPEG is None:
    raise RuntimeError(
        "หา ffmpeg ไม่เจอใน PATH — ตรวจ ffmpeg ก่อนรัน ASR"
    )

if not RAW_VIDEO.exists():
    raise FileNotFoundError(RAW_VIDEO)

try:
    from faster_whisper import WhisperModel
    print("faster-whisper: READY")
except ImportError:
    print("faster-whisper: NOT INSTALLED")
    print("ถ้าต้องการรัน Optional ASR ให้กลับไป Cell 12")



## Cell 14 — ASR Config

เริ่มด้วย `small` ก่อนสำหรับ RTX 4050 Laptop 6GB

ถ้า VRAM ไม่พอ:
- `DEVICE = "cpu"`
- `COMPUTE_TYPE = "int8"`


In [ ]:

WHISPER_MODEL_SIZE = "small"
WHISPER_DEVICE = "cuda"
WHISPER_COMPUTE_TYPE = "float16"
WHISPER_LANGUAGE = "th"

print("Model       :", WHISPER_MODEL_SIZE)
print("Device      :", WHISPER_DEVICE)
print("Compute type:", WHISPER_COMPUTE_TYPE)
print("Cache       :", WHISPER_CACHE)



## Cell 15 — Load Whisper Model

ครั้งแรกจะดาวน์โหลด model ลง `D:\Face_detection\cache\faster_whisper`


In [ ]:

from faster_whisper import WhisperModel

whisper_model = WhisperModel(
    WHISPER_MODEL_SIZE,
    device=WHISPER_DEVICE,
    compute_type=WHISPER_COMPUTE_TYPE,
    download_root=str(WHISPER_CACHE),
)

print("✅ Whisper model loaded")



## Cell 16 — Extract Segment Audio + Timed ASR Tokens

เพื่อไม่ต้อง transcribe ทั้งวิดีโอซ้ำหลายครั้ง:
- ใช้แต่ละ sentence segment เดิม
- ตัด audio ช่วงนั้นด้วย ffmpeg
- ASR พร้อม `word_timestamps=True`
- เวลา ASR จะถูกแปลงกลับเป็น **absolute video time**

> รอบแรกแนะนำทดลอง `MAX_ASR_SEGMENTS = 20` ก่อน  
> ถ้าผลโอเคค่อยเปลี่ยนเป็น `None`


In [ ]:

import subprocess
import tempfile
from pathlib import Path

MAX_ASR_SEGMENTS = 20   # ทดลองก่อน; เปลี่ยนเป็น None เมื่อต้องการรันทั้งหมด

def extract_audio_segment(video_path, start_time, end_time, out_wav):
    duration = max(0.05, float(end_time) - float(start_time))

    cmd = [
        "ffmpeg",
        "-y",
        "-loglevel", "error",
        "-ss", f"{float(start_time):.3f}",
        "-i", str(video_path),
        "-t", f"{duration:.3f}",
        "-vn",
        "-ac", "1",
        "-ar", "16000",
        "-c:a", "pcm_s16le",
        str(out_wav),
    ]

    subprocess.run(cmd, check=True)

def transcribe_segment(row):
    tmp_wav = TMP_ROOT / f"{row['clip_id']}.wav"

    extract_audio_segment(
        RAW_VIDEO,
        row["start_time"],
        row["end_time"],
        tmp_wav,
    )

    segments, info = whisper_model.transcribe(
        str(tmp_wav),
        language=WHISPER_LANGUAGE,
        beam_size=5,
        word_timestamps=True,
        vad_filter=False,
        condition_on_previous_text=False,
    )

    results = []

    for seg in segments:
        if seg.words is None:
            continue

        for wi, w in enumerate(seg.words):
            token_text = str(w.word).strip()

            if not token_text:
                continue

            results.append({
                "clip_id": row["clip_id"],
                "speaker_id": int(row["speaker_id"]),
                "split": row["split"],
                "sentence_text": row["text"],
                "sentence_start": float(row["start_time"]),
                "sentence_end": float(row["end_time"]),
                "asr_token_index": wi,
                "asr_token": token_text,
                "asr_start": float(row["start_time"]) + float(w.start),
                "asr_end": float(row["start_time"]) + float(w.end),
                "asr_probability": (
                    float(w.probability)
                    if getattr(w, "probability", None) is not None
                    else np.nan
                ),
            })

    try:
        tmp_wav.unlink()
    except Exception:
        pass

    return results

asr_source = tokenized_df.copy()

if MAX_ASR_SEGMENTS is not None:
    asr_source = asr_source.head(MAX_ASR_SEGMENTS)

print("ASR segments to process:", len(asr_source))


## Cell 17 — Run ASR

In [ ]:

from time import time

asr_rows = []
t0 = time()

for i, (_, row) in enumerate(asr_source.iterrows(), start=1):
    try:
        rows = transcribe_segment(row)
        asr_rows.extend(rows)
        print(
            f"[{i:4d}/{len(asr_source)}] "
            f"{row['clip_id']} | tokens={len(rows)}"
        )
    except Exception as e:
        print(
            f"[ERROR] {row['clip_id']} | "
            f"{type(e).__name__}: {e}"
        )

asr_df = pd.DataFrame(asr_rows)

ASR_CSV = OUTPUT_ROOT / "asr_word_timestamps.csv"
asr_df.to_csv(ASR_CSV, index=False, encoding="utf-8-sig")

print("=" * 80)
print("ASR FINISHED")
print("=" * 80)
print("Timed ASR tokens:", len(asr_df))
print("Minutes         :", round((time() - t0) / 60, 2))
print("Saved           :", ASR_CSV)

display(asr_df.head(50))



## Cell 18 — Conservative Word ↔ ASR Matching

หลักการรอบแรก:
- อยู่ใน `clip_id` เดียวกัน
- normalize แล้ว **ต้องตรงกัน**
- ถ้าคำเดียวกันปรากฏหลายครั้งใน sentence จะจับตามลำดับ occurrence
- เก็บ probability ของ Whisper
- ถ้าจับไม่ได้ = `review`

เรา **ไม่บังคับ match** เพื่อป้องกัน label ผิด


In [ ]:
# ============================================================
# CELL 18 V3 — Character-position alignment
# Subtitle words -> continuous sentence text
# Whisper subwords -> continuous ASR text + timestamp mapping
# แล้ว map target word span กลับไปเป็นเวลา
# ============================================================

import re
import unicodedata
import numpy as np
import pandas as pd
from difflib import SequenceMatcher


# ------------------------------------------------------------
# 1) Normalize text
# ------------------------------------------------------------

def normalize_for_alignment(text):
    """
    Normalize เพื่อใช้ alignment

    - NFC unicode
    - ลบ whitespace
    - เก็บไทย อังกฤษ ตัวเลข
    - ลบ punctuation
    """
    text = unicodedata.normalize("NFC", str(text))

    text = re.sub(r"\s+", "", text)

    text = re.sub(
        r"[^\u0E00-\u0E7FA-Za-z0-9]",
        "",
        text
    )

    return text.lower().strip()


# ------------------------------------------------------------
# 2) รวม Whisper tokens เป็น string ต่อเนื่อง
#    พร้อม map "ตัวอักษรแต่ละตำแหน่ง" -> timestamp
# ------------------------------------------------------------

def build_asr_character_map(asr_tokens):

    asr_text = ""

    char_map = []

    for token_idx, token in enumerate(asr_tokens):

        raw_piece = str(token["asr_token"])

        piece = normalize_for_alignment(raw_piece)

        if not piece:
            continue

        start = float(token["asr_start"])
        end = float(token["asr_end"])

        probability = token.get(
            "asr_probability",
            np.nan
        )

        if pd.notna(probability):
            probability = float(probability)

        # ----------------------------------------------------
        # ถ้า token มีหลายตัวอักษร
        # กระจายเวลาโดยประมาณภายใน token
        #
        # เช่น:
        # "เพ" 50.76-50.86
        #
        # เรายังรู้แค่ timestamp ระดับ token
        # จึงแบ่ง span ภายใน token เพื่อ map ตัวอักษร
        # ----------------------------------------------------

        n = len(piece)

        duration = max(
            0.0,
            end - start
        )

        for j, ch in enumerate(piece):

            if duration > 0 and n > 0:

                ch_start = (
                    start
                    + duration * (j / n)
                )

                ch_end = (
                    start
                    + duration * ((j + 1) / n)
                )

            else:

                ch_start = start
                ch_end = end

            asr_text += ch

            char_map.append({
                "char": ch,
                "start": ch_start,
                "end": ch_end,
                "token_start": start,
                "token_end": end,
                "token_index": token_idx,
                "token_text": raw_piece,
                "probability": probability,
            })

    return asr_text, char_map


# ------------------------------------------------------------
# 3) หา character index ของ target word
#    ในข้อความ sentence ที่ normalize แล้ว
# ------------------------------------------------------------

def build_target_word_spans(words):

    spans = []

    current_pos = 0

    for original_index, word in words:

        norm_word = normalize_for_alignment(word)

        if not norm_word:
            continue

        start_pos = current_pos

        end_pos = (
            current_pos
            + len(norm_word)
        )

        spans.append({
            "word_index": int(original_index),
            "word": word,
            "norm_word": norm_word,
            "target_start": start_pos,
            "target_end": end_pos,
        })

        current_pos = end_pos

    target_text = "".join(
        x["norm_word"]
        for x in spans
    )

    return target_text, spans


# ------------------------------------------------------------
# 4) Character-level sequence alignment
# ------------------------------------------------------------

def create_character_alignment(
    target_text,
    asr_text
):

    matcher = SequenceMatcher(
        None,
        target_text,
        asr_text,
        autojunk=False
    )

    target_to_asr = {}

    for block in matcher.get_matching_blocks():

        # block:
        # a = target start
        # b = asr start
        # size = matching length

        for k in range(block.size):

            target_pos = block.a + k
            asr_pos = block.b + k

            target_to_asr[target_pos] = asr_pos

    similarity = matcher.ratio()

    return target_to_asr, similarity


# ------------------------------------------------------------
# 5) Map word -> timestamp
# ------------------------------------------------------------

def align_words_with_character_map(
    target_words,
    asr_tokens
):

    # target_words:
    # [(real_word_index, word), ...]

    target_text, word_spans = (
        build_target_word_spans(
            target_words
        )
    )

    asr_text, char_map = (
        build_asr_character_map(
            asr_tokens
        )
    )

    target_to_asr, sentence_similarity = (
        create_character_alignment(
            target_text,
            asr_text
        )
    )

    results = []

    for span in word_spans:

        word_start_pos = (
            span["target_start"]
        )

        word_end_pos = (
            span["target_end"]
        )

        matched_asr_positions = []

        matched_chars = 0

        total_chars = max(
            1,
            word_end_pos
            - word_start_pos
        )

        # ----------------------------------------------------
        # ดู character ของ target word ว่าตัวไหน match
        # กับ ASR character บ้าง
        # ----------------------------------------------------

        for target_pos in range(
            word_start_pos,
            word_end_pos
        ):

            if target_pos in target_to_asr:

                asr_pos = (
                    target_to_asr[
                        target_pos
                    ]
                )

                if (
                    0
                    <= asr_pos
                    < len(char_map)
                ):
                    matched_asr_positions.append(
                        asr_pos
                    )

                    matched_chars += 1

        char_coverage = (
            matched_chars
            / total_chars
        )

        # ----------------------------------------------------
        # ถ้าไม่มี match เลย
        # ----------------------------------------------------

        if not matched_asr_positions:

            results.append({
                "word_index":
                    span["word_index"],

                "word":
                    span["word"],

                "word_start":
                    np.nan,

                "word_end":
                    np.nan,

                "word_duration":
                    np.nan,

                "alignment_status":
                    "review",

                "alignment_confidence":
                    0.0,

                "char_coverage":
                    0.0,

                "sentence_similarity":
                    sentence_similarity,

                "matched_asr_text":
                    "",

                "matched_char_count":
                    0,

                "total_char_count":
                    total_chars,
            })

            continue

        # ----------------------------------------------------
        # ใช้ min/max ASR char position ของ word
        # ----------------------------------------------------

        first_pos = min(
            matched_asr_positions
        )

        last_pos = max(
            matched_asr_positions
        )

        selected_chars = (
            char_map[
                first_pos:
                last_pos + 1
            ]
        )

        word_start = float(
            selected_chars[0][
                "token_start"
            ]
        )

        word_end = float(
            selected_chars[-1][
                "token_end"
            ]
        )

        word_duration = (
            word_end
            - word_start
        )

        matched_asr_text = "".join(
            x["char"]
            for x in selected_chars
        )

        probs = [
            x["probability"]
            for x in selected_chars
            if pd.notna(
                x["probability"]
            )
        ]

        mean_probability = (
            float(np.mean(probs))
            if probs
            else np.nan
        )

        # ----------------------------------------------------
        # confidence = character coverage
        # ผสมกับ Whisper probability
        # ----------------------------------------------------

        if pd.notna(mean_probability):

            alignment_confidence = (
                0.7 * char_coverage
                + 0.3 * mean_probability
            )

        else:

            alignment_confidence = (
                char_coverage
            )

        # ----------------------------------------------------
        # status
        # ----------------------------------------------------

        if char_coverage >= 0.80:

            status = (
                "character_aligned"
            )

        elif char_coverage >= 0.60:

            status = (
                "partial_alignment"
            )

        else:

            status = "review"

        results.append({
            "word_index":
                span["word_index"],

            "word":
                span["word"],

            "word_start":
                word_start,

            "word_end":
                word_end,

            "word_duration":
                word_duration,

            "alignment_status":
                status,

            "alignment_confidence":
                alignment_confidence,

            "char_coverage":
                char_coverage,

            "sentence_similarity":
                sentence_similarity,

            "matched_asr_text":
                matched_asr_text,

            "matched_char_count":
                matched_chars,

            "total_char_count":
                total_chars,
        })

    return (
        results,
        target_text,
        asr_text,
        sentence_similarity
    )


# ============================================================
# 6) Run alignment
# ============================================================

aligned_rows = []

available_clips = set(
    asr_df["clip_id"].unique()
)

for clip_id, group in (
    v5_occ.groupby("clip_id")
):

    if clip_id not in available_clips:
        continue

    group = (
        group
        .sort_values("word_index")
        .reset_index(drop=True)
    )

    # --------------------------------------------------------
    # IMPORTANT:
    # ใช้ word_index จริงจาก dataset
    # --------------------------------------------------------

    target_words = [
        (
            int(row["word_index"]),
            row["word"]
        )
        for _, row in group.iterrows()
    ]

    # --------------------------------------------------------
    # ASR token ของ clip นี้
    # --------------------------------------------------------

    asr_group = (
        asr_df[
            asr_df["clip_id"]
            == clip_id
        ]
        .sort_values(
            [
                "asr_start",
                "asr_token_index",
            ]
        )
        .reset_index(drop=True)
    )

    asr_tokens = (
        asr_group
        .to_dict("records")
    )

    (
        results,
        target_text,
        asr_text,
        sentence_similarity,
    ) = align_words_with_character_map(
        target_words,
        asr_tokens
    )

    # --------------------------------------------------------
    # map original row ด้วย word_index จริง
    # --------------------------------------------------------

    original_map = {
        int(row["word_index"]):
            row.to_dict()

        for _, row
        in group.iterrows()
    }

    for result in results:

        real_index = int(
            result["word_index"]
        )

        if (
            real_index
            not in original_map
        ):
            continue

        original = (
            original_map[
                real_index
            ].copy()
        )

        original.update(result)

        original["target_sentence_normalized"] = (
            target_text
        )

        original["asr_sentence_normalized"] = (
            asr_text
        )

        original["alignment_source"] = (
            "character_position_alignment_v3"
        )

        aligned_rows.append(
            original
        )


# ============================================================
# 7) DataFrame + Save
# ============================================================

alignment_df = pd.DataFrame(
    aligned_rows
)

ALIGN_V3_CSV = (
    OUTPUT_ROOT
    / "word_alignment_candidates_v3.csv"
)

alignment_df.to_csv(
    ALIGN_V3_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 8) Summary
# ============================================================

print("=" * 90)
print(
    "CHARACTER ALIGNMENT V3 FINISHED"
)
print("=" * 90)

print(
    alignment_df[
        "alignment_status"
    ].value_counts(
        dropna=False
    )
)

total = len(
    alignment_df
)

good = int(
    (
        alignment_df[
            "alignment_status"
        ]
        == "character_aligned"
    ).sum()
)

partial = int(
    (
        alignment_df[
            "alignment_status"
        ]
        == "partial_alignment"
    ).sum()
)

review = int(
    (
        alignment_df[
            "alignment_status"
        ]
        == "review"
    ).sum()
)

print()
print(
    "Total           :",
    total
)

print(
    "Character align :",
    good
)

print(
    "Partial         :",
    partial
)

print(
    "Review          :",
    review
)

if total > 0:

    print(
        "Good rate       :",
        f"{good/total*100:.2f}%"
    )

    print(
        "Good+Partial    :",
        f"{(good+partial)/total*100:.2f}%"
    )


print()
print(
    "Sentence similarity:"
)

display(
    alignment_df[
        "sentence_similarity"
    ].describe()
)


print()
print(
    "Saved:"
)

print(
    ALIGN_V3_CSV
)


# ============================================================
# 9) Preview
# ============================================================

display(
    alignment_df[
        [
            "clip_id",
            "word_index",
            "word",
            "matched_asr_text",

            "word_start",
            "word_end",
            "word_duration",

            "char_coverage",
            "alignment_confidence",

            "sentence_similarity",
            "alignment_status",
        ]
    ].head(100)
)


## Cell 19 — Filter Candidate ที่ใช้สร้าง Word Clip ได้

รอบแรกให้ conservative:
- `alignment_status == exact_asr_token`
- duration > 0
- timestamp อยู่ใน sentence boundary
- confidence ผ่าน threshold

ค่า threshold เริ่มต้น `0.50`  
**ยังไม่ใช่ค่าตายตัว** ต้องดู QA ก่อน


In [ ]:
# CELL 19 — Strict filter สำหรับสร้าง word clips

MIN_CHAR_COVERAGE = 0.80
MIN_ALIGNMENT_CONFIDENCE = 0.70

MIN_WORD_DURATION = 0.04
MAX_WORD_DURATION = 2.50


candidate_df = alignment_df.copy()

candidate_df["inside_sentence"] = (
    (candidate_df["word_start"] >= candidate_df["sentence_start"] - 0.10)
    &
    (candidate_df["word_end"] <= candidate_df["sentence_end"] + 0.10)
)

candidate_df["valid_duration"] = (
    candidate_df["word_duration"].between(
        MIN_WORD_DURATION,
        MAX_WORD_DURATION
    )
)

candidate_df["alignment_ok"] = (
    (candidate_df["alignment_status"] == "character_aligned")
    &
    (candidate_df["char_coverage"] >= MIN_CHAR_COVERAGE)
    &
    (candidate_df["alignment_confidence"] >= MIN_ALIGNMENT_CONFIDENCE)
    &
    candidate_df["inside_sentence"]
    &
    candidate_df["valid_duration"]
)

total = len(candidate_df)
usable = int(candidate_df["alignment_ok"].sum())

print("=" * 80)
print("STRICT ALIGNMENT FILTER")
print("=" * 80)

print("Total :", total)
print("Usable:", usable)

if total:
    print(f"Usable rate: {usable/total*100:.2f}%")

print("\nRejected reasons:")
print(
    candidate_df.loc[
        ~candidate_df["alignment_ok"],
        [
            "word",
            "alignment_status",
            "char_coverage",
            "alignment_confidence",
            "word_duration"
        ]
    ]
)

print("\nAccepted examples:")
display(
    candidate_df[
        candidate_df["alignment_ok"]
    ][
        [
            "clip_id",
            "word",
            "matched_asr_text",
            "word_start",
            "word_end",
            "word_duration",
            "char_coverage",
            "alignment_confidence"
        ]
    ].head(100)
)


## Cell 20 — Final QA Summary

ก่อนสร้าง V5 word clips ให้ตรวจ:
1. คำที่ tokenizer ตัดออกมาถูกหรือไม่
2. คำยอดนิยมมีจำนวนจริงพอหรือไม่
3. ASR timestamp ตรงกับปากจริงหรือไม่
4. duration ของแต่ละคำสมเหตุผลหรือไม่
5. อย่าใช้แถว `review` ไป train

ขั้นต่อไปหลัง Notebook นี้ผ่าน:

```text
word_train_ready_candidates.csv
        ↓
ตัด Face + Mouth frames ตาม word_start / word_end
        ↓
Word-level clips
        ↓
V5 Word Classification
        ↓
CrossEntropyLoss
        ↓
Top-1 / Top-5 Accuracy + Confusion Matrix
```

### แนะนำการทดลอง V5 รอบแรก

เริ่มเล็กก่อน:
- 20–50 คำ
- เลือกคำที่มี train examples มากที่สุด
- พยายามให้แต่ละคำมี ≥10–20 train examples
- ตรวจคลิปคำด้วยตาก่อน train

**ยังไม่ต้องกลับไป CTC ใน V5 รอบแรก**


In [ ]:
# ============================================================
# CELL 20 — Final Word Manifest V3
# สร้างช่วงเวลาสำหรับตัดคลิปคำจริง
# ============================================================

import numpy as np
import pandas as pd


# ------------------------------------------------------------
# Context รอบคำ
# ------------------------------------------------------------

LEFT_CONTEXT_SEC = 0.10
RIGHT_CONTEXT_SEC = 0.10

# คลาสที่ไม่ต้องการใช้เป็นคำเดี่ยว
EXCLUDED_WORDS = {
    "ๆ",
}


# ------------------------------------------------------------
# เลือกเฉพาะ alignment ที่ผ่าน strict filter
# ------------------------------------------------------------

train_ready_words = candidate_df[
    candidate_df["alignment_ok"]
].copy()


# ------------------------------------------------------------
# ตัดคำที่ไม่เหมาะเป็น standalone class
# ------------------------------------------------------------

train_ready_words = train_ready_words[
    ~train_ready_words["word"].isin(EXCLUDED_WORDS)
].copy()


# ------------------------------------------------------------
# เก็บ timestamp ดิบไว้
# ------------------------------------------------------------

train_ready_words["raw_word_start"] = (
    train_ready_words["word_start"]
)

train_ready_words["raw_word_end"] = (
    train_ready_words["word_end"]
)


# ------------------------------------------------------------
# เพิ่ม context ก่อน/หลังคำ
#
# ตัวอย่าง:
# raw = 51.36–51.50
#
# clip = 51.26–51.60
#
# ช่วยเก็บการเริ่ม/หยุดเคลื่อนไหวของปาก
# ------------------------------------------------------------

train_ready_words["clip_start"] = (
    train_ready_words["raw_word_start"]
    - LEFT_CONTEXT_SEC
)

train_ready_words["clip_end"] = (
    train_ready_words["raw_word_end"]
    + RIGHT_CONTEXT_SEC
)


# ------------------------------------------------------------
# ไม่ให้ออกนอกขอบเขตประโยค
# ------------------------------------------------------------

train_ready_words["clip_start"] = np.maximum(
    train_ready_words["clip_start"],
    train_ready_words["sentence_start"]
)

train_ready_words["clip_end"] = np.minimum(
    train_ready_words["clip_end"],
    train_ready_words["sentence_end"]
)


# ------------------------------------------------------------
# ความยาวคลิปจริง
# ------------------------------------------------------------

train_ready_words["clip_duration"] = (
    train_ready_words["clip_end"]
    - train_ready_words["clip_start"]
)


# ------------------------------------------------------------
# ID ของแต่ละ word clip
#
# เช่น
# video001_seg000010_spk000_word002
# ------------------------------------------------------------

train_ready_words["word_clip_id"] = (
    train_ready_words.apply(
        lambda r:
        f"{r['clip_id']}_word{int(r['word_index']):03d}",
        axis=1
    )
)


# ------------------------------------------------------------
# เรียงข้อมูล
# ------------------------------------------------------------

train_ready_words = (
    train_ready_words
    .sort_values(
        [
            "split",
            "clip_id",
            "word_index",
        ]
    )
    .reset_index(drop=True)
)


# ------------------------------------------------------------
# Safety checks
# ------------------------------------------------------------

assert (
    train_ready_words["clip_duration"] > 0
).all(), "พบ clip_duration <= 0"

assert train_ready_words[
    "clip_start"
].notna().all()

assert train_ready_words[
    "clip_end"
].notna().all()


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

FINAL_CSV = (
    OUTPUT_ROOT
    / "word_train_ready_candidates_v3.csv"
)

train_ready_words.to_csv(
    FINAL_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# SUMMARY
# ============================================================

print("=" * 85)
print("FINAL WORD MANIFEST V3")
print("=" * 85)

print(
    "Total usable word clips :",
    len(train_ready_words)
)

print(
    "Unique word classes     :",
    train_ready_words["word"].nunique()
)

print(
    "Min clip duration       :",
    round(
        train_ready_words["clip_duration"].min(),
        3
    ),
    "sec"
)

print(
    "Mean clip duration      :",
    round(
        train_ready_words["clip_duration"].mean(),
        3
    ),
    "sec"
)

print(
    "Max clip duration       :",
    round(
        train_ready_words["clip_duration"].max(),
        3
    ),
    "sec"
)


print("\nBy split:")
print(
    train_ready_words[
        "split"
    ].value_counts()
)


print("\nTop words:")
print(
    train_ready_words[
        "word"
    ].value_counts().head(30)
)


print("\nSaved:")
print(FINAL_CSV)


# ------------------------------------------------------------
# Preview
# ------------------------------------------------------------

display(
    train_ready_words[
        [
            "word_clip_id",
            "word",
            "split",

            "raw_word_start",
            "raw_word_end",

            "clip_start",
            "clip_end",
            "clip_duration",

            "char_coverage",
            "alignment_confidence",
        ]
    ].head(100)
)

In [ ]:
# ============================================================
# CELL 21 — Export Word Video Clips
# ตัดคลิปวิดีโอจริงจากช่วงเวลา clip_start / clip_end
# ============================================================

from pathlib import Path
import subprocess
import shutil
import pandas as pd


# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

PROJECT_ROOT = Path(r"D:\Face_detection")
VIDEO_ID = "video001"

SOURCE_VIDEO = (
    PROJECT_ROOT
    / "dataset"
    / "raw_videos"
    / f"{VIDEO_ID}.webm"
)

WORD_CLIP_ROOT = (
    PROJECT_ROOT
    / "dataset"
    / "word_level_v5"
    / VIDEO_ID
    / "clips_v3"
)

WORD_CLIP_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# CHECK FFMPEG
# ------------------------------------------------------------

ffmpeg_path = shutil.which("ffmpeg")

if ffmpeg_path is None:
    raise RuntimeError(
        "ไม่พบ ffmpeg ใน PATH"
    )

if not SOURCE_VIDEO.exists():
    raise FileNotFoundError(
        f"ไม่พบวิดีโอ:\n{SOURCE_VIDEO}"
    )

print("FFmpeg :", ffmpeg_path)
print("Video  :", SOURCE_VIDEO)
print("Output :", WORD_CLIP_ROOT)


# ============================================================
# Helper — ทำชื่อโฟลเดอร์คำให้ปลอดภัย
# ============================================================

def safe_word_name(word):

    word = str(word).strip()

    bad_chars = [
        '\\', '/', ':', '*',
        '?', '"', '<', '>', '|'
    ]

    for ch in bad_chars:
        word = word.replace(ch, "_")

    return word


# ============================================================
# Helper — export 1 clip
# ============================================================

def export_word_clip(
    input_video,
    output_path,
    start_time,
    end_time
):

    duration = float(
        end_time - start_time
    )

    if duration <= 0:
        return False, "duration <= 0"

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    command = [
        ffmpeg_path,

        "-y",

        # เวลาเริ่ม
        "-ss",
        f"{float(start_time):.3f}",

        # input
        "-i",
        str(input_video),

        # ความยาว
        "-t",
        f"{duration:.3f}",

        # ไม่มีเสียง
        # เพราะโมเดลอ่านปากใช้ visual only
        "-an",

        # codec
        "-c:v",
        "libx264",

        # preset เร็ว
        "-preset",
        "fast",

        # คุณภาพ
        "-crf",
        "20",

        # pixel format ใช้งานง่าย
        "-pix_fmt",
        "yuv420p",

        # ป้องกัน log ยาว
        "-loglevel",
        "error",

        str(output_path),
    ]

    try:

        result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            timeout=60
        )

        if result.returncode != 0:

            return (
                False,
                result.stderr.strip()
            )

        if not output_path.exists():

            return (
                False,
                "output file not created"
            )

        if output_path.stat().st_size == 0:

            return (
                False,
                "output file is empty"
            )

        return True, ""

    except Exception as e:

        return False, str(e)


# ============================================================
# EXPORT
# ============================================================

export_rows = []

total = len(train_ready_words)

success_count = 0
fail_count = 0


for idx, row in train_ready_words.iterrows():

    word = str(
        row["word"]
    ).strip()

    split = str(
        row["split"]
    ).strip()

    safe_word = safe_word_name(
        word
    )

    clip_id = str(
        row["word_clip_id"]
    )

    start_time = float(
        row["clip_start"]
    )

    end_time = float(
        row["clip_end"]
    )

    # --------------------------------------------------------
    # Directory
    #
    # clips_v3/
    # ├── train/
    # │   └── ไม่/
    # │       └── xxxx.mp4
    # --------------------------------------------------------

    output_dir = (
        WORD_CLIP_ROOT
        / split
        / safe_word
    )

    output_file = (
        output_dir
        / f"{clip_id}.mp4"
    )

    success, error = export_word_clip(
        SOURCE_VIDEO,
        output_file,
        start_time,
        end_time
    )

    if success:
        success_count += 1
        status = "success"
    else:
        fail_count += 1
        status = "failed"

    export_row = row.to_dict()

    export_row.update({
        "export_status": status,
        "video_clip_path": str(
            output_file
        ),
        "export_error": error,
    })

    export_rows.append(
        export_row
    )

    # progress
    if (
        (idx + 1) % 10 == 0
        or idx == total - 1
    ):

        print(
            f"[{idx+1}/{total}] "
            f"success={success_count} "
            f"failed={fail_count}"
        )


# ============================================================
# SAVE EXPORT MANIFEST
# ============================================================

export_df = pd.DataFrame(
    export_rows
)

EXPORT_MANIFEST = (
    PROJECT_ROOT
    / "dataset"
    / "word_level_v5"
    / VIDEO_ID
    / "word_clip_export_manifest_v3.csv"
)

export_df.to_csv(
    EXPORT_MANIFEST,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 85)
print("WORD CLIP EXPORT FINISHED")
print("=" * 85)

print(
    "Total   :",
    len(export_df)
)

print(
    "Success :",
    success_count
)

print(
    "Failed  :",
    fail_count
)

if len(export_df) > 0:

    print(
        "Success rate:",
        f"{success_count / len(export_df) * 100:.2f}%"
    )


print("\nOutput root:")
print(
    WORD_CLIP_ROOT
)

print("\nManifest:")
print(
    EXPORT_MANIFEST
)


# ============================================================
# Preview
# ============================================================

display(
    export_df[
        [
            "word_clip_id",
            "word",
            "split",

            "clip_start",
            "clip_end",
            "clip_duration",

            "export_status",
            "video_clip_path",
        ]
    ].head(100)
)

In [ ]:
# ============================================================
# CELL 22 — QA Word Clips
# สุ่มคลิปมาตรวจว่า "ภาพปากตรงกับ label จริงไหม"
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np

# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

QA_SAMPLE_SIZE = 30
QA_RANDOM_SEED = 42

# ------------------------------------------------------------
# เลือกเฉพาะไฟล์ที่ export สำเร็จ
# ------------------------------------------------------------

qa_source = export_df[
    export_df["export_status"] == "success"
].copy()

# เช็กว่าไฟล์มีอยู่จริง
qa_source["file_exists"] = qa_source[
    "video_clip_path"
].apply(
    lambda p: Path(p).exists()
)

qa_source = qa_source[
    qa_source["file_exists"]
].copy()

# ------------------------------------------------------------
# สุ่มแบบพยายามกระจายหลายคำ
# ------------------------------------------------------------

rng = np.random.default_rng(
    QA_RANDOM_SEED
)

sample_parts = []

# เอาตัวอย่างจากแต่ละคำก่อน
for word, group in qa_source.groupby("word"):

    n = min(
        len(group),
        2
    )

    sampled = group.sample(
        n=n,
        random_state=QA_RANDOM_SEED
    )

    sample_parts.append(
        sampled
    )

if sample_parts:

    qa_sample = pd.concat(
        sample_parts,
        ignore_index=True
    )

else:

    qa_sample = qa_source.copy()

# ถ้าเกินจำนวนที่กำหนด ให้สุ่มลด
if len(qa_sample) > QA_SAMPLE_SIZE:

    qa_sample = qa_sample.sample(
        n=QA_SAMPLE_SIZE,
        random_state=QA_RANDOM_SEED
    )

# ถ้ายังไม่ครบ เติมจากข้อมูลที่เหลือ
elif len(qa_sample) < QA_SAMPLE_SIZE:

    remaining = qa_source[
        ~qa_source["word_clip_id"].isin(
            qa_sample["word_clip_id"]
        )
    ]

    need = (
        QA_SAMPLE_SIZE
        - len(qa_sample)
    )

    if len(remaining) > 0:

        extra = remaining.sample(
            n=min(
                need,
                len(remaining)
            ),
            random_state=QA_RANDOM_SEED
        )

        qa_sample = pd.concat(
            [
                qa_sample,
                extra
            ],
            ignore_index=True
        )

# ------------------------------------------------------------
# จัดเรียง
# ------------------------------------------------------------

qa_sample = (
    qa_sample
    .sort_values(
        [
            "word",
            "clip_id",
        ]
    )
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# เพิ่มช่องสำหรับตรวจด้วยตา
#
# qa_result:
#   correct
#   uncertain
#   wrong
#
# qa_note:
#   หมายเหตุ เช่น
#   "ต้นคำโดนตัด"
#   "มีคำถัดไปติดมา"
# ------------------------------------------------------------

qa_sample["qa_result"] = ""
qa_sample["qa_note"] = ""

# ------------------------------------------------------------
# SAVE QA CSV
# ------------------------------------------------------------

QA_CSV = (
    PROJECT_ROOT
    / "dataset"
    / "word_level_v5"
    / VIDEO_ID
    / "word_clip_qa_sample_v3.csv"
)

qa_sample.to_csv(
    QA_CSV,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# SUMMARY
# ------------------------------------------------------------

print("=" * 85)
print("WORD CLIP QA SAMPLE")
print("=" * 85)

print(
    "Available clips :",
    len(qa_source)
)

print(
    "QA sample       :",
    len(qa_sample)
)

print(
    "Unique words    :",
    qa_sample["word"].nunique()
)

print("\nQA file:")
print(
    QA_CSV
)

print("\nให้เปิด video_clip_path ทีละไฟล์แล้วตรวจ:")
print("correct   = คำตรงและช่วงปากดี")
print("uncertain = ไม่แน่ใจ / ช่วงสั้นเกิน")
print("wrong     = ไม่ตรงคำ / ตัดผิดช่วง")

# ------------------------------------------------------------
# TABLE
# ------------------------------------------------------------

display(
    qa_sample[
        [
            "word",
            "word_clip_id",

            "raw_word_start",
            "raw_word_end",

            "clip_start",
            "clip_end",
            "clip_duration",

            "char_coverage",
            "alignment_confidence",

            "video_clip_path",

            "qa_result",
            "qa_note",
        ]
    ]
)

In [ ]:
# ============================================================
# CELL 23 — FULL ASR FOR ALL 732 SENTENCES + RESUME
# ============================================================

from pathlib import Path
import pandas as pd
import numpy as np
from tqdm.auto import tqdm
import traceback


# ------------------------------------------------------------
# CHECK VARIABLES
# ------------------------------------------------------------

required_vars = [
    "tokenized_df",
    "whisper_model",
    "RAW_VIDEO",
    "TMP_ROOT",
    "OUTPUT_ROOT",
]

for name in required_vars:
    if name not in globals():
        raise RuntimeError(
            f"ไม่พบตัวแปร {name} "
            f"กรุณารัน Cell ก่อนหน้าที่สร้างตัวแปรนี้ก่อน"
        )


print("=" * 90)
print("FULL ASR — ALL SENTENCES")
print("=" * 90)

print("Total sentence clips :", len(tokenized_df))
print("Raw video            :", RAW_VIDEO)
print("Output               :", OUTPUT_ROOT)


# ============================================================
# OUTPUT FILES
# ============================================================

FULL_ASR_CSV = (
    OUTPUT_ROOT /
    "asr_word_timestamps_full.csv"
)

ASR_PROGRESS_CSV = (
    OUTPUT_ROOT /
    "asr_completed_clips.csv"
)

ASR_ERROR_CSV = (
    OUTPUT_ROOT /
    "asr_errors.csv"
)


# ============================================================
# LOAD OLD RESULTS FOR RESUME
# ============================================================

if FULL_ASR_CSV.exists():

    old_asr_df = pd.read_csv(
        FULL_ASR_CSV
    )

    print(
        "พบ ASR cache เดิม:",
        len(old_asr_df),
        "tokens"
    )

else:

    old_asr_df = pd.DataFrame()


if ASR_PROGRESS_CSV.exists():

    completed_df = pd.read_csv(
        ASR_PROGRESS_CSV
    )

    completed_clips = set(
        completed_df["clip_id"]
        .astype(str)
        .tolist()
    )

else:

    completed_clips = set()


print(
    "Completed sentence clips:",
    len(completed_clips)
)


# ============================================================
# IMPORTANT:
# ใช้ ALL 732 — ไม่มี .head(20)
# ============================================================

asr_source_full = (
    tokenized_df
    .sort_values(
        [
            "split",
            "start_time",
        ]
    )
    .reset_index(drop=True)
)

remaining_df = asr_source_full[
    ~asr_source_full[
        "clip_id"
    ].astype(str).isin(
        completed_clips
    )
].copy()


print(
    "Remaining sentence clips:",
    len(remaining_df)
)


# ============================================================
# TRANSCRIBE
# ใช้ฟังก์ชัน transcribe_segment จาก Cell 16 เดิม
# ============================================================

if "transcribe_segment" not in globals():

    raise RuntimeError(
        "ไม่พบฟังก์ชัน transcribe_segment "
        "กรุณารัน Cell 16 เดิมก่อน"
    )


all_new_rows = []

progress_rows = []

error_rows = []


# ------------------------------------------------------------
# Save interval
# ทุก 10 ประโยค save ลง disk
# ------------------------------------------------------------

SAVE_EVERY = 10


for loop_index, (_, row) in enumerate(
    tqdm(
        remaining_df.iterrows(),
        total=len(remaining_df),
        desc="Full ASR"
    ),
    start=1
):

    clip_id = str(
        row["clip_id"]
    )

    try:

        result_rows = (
            transcribe_segment(row)
        )

        # เก็บ token ทั้งหมด
        if result_rows:
            all_new_rows.extend(
                result_rows
            )

        # ถึงแม้ Whisper ไม่ได้ token
        # ก็ถือว่าประมวลผล clip แล้ว
        progress_rows.append({
            "clip_id": clip_id,
            "status": "completed",
            "num_asr_tokens": len(
                result_rows
            ),
        })

    except Exception as e:

        error_rows.append({
            "clip_id": clip_id,
            "split": row["split"],
            "start_time":
                row["start_time"],
            "end_time":
                row["end_time"],
            "error_type":
                type(e).__name__,
            "error":
                str(e),
        })

        print(
            f"\n[ERROR] {clip_id}"
        )

        print(
            type(e).__name__,
            ":",
            e
        )


    # ========================================================
    # SAVE CHECKPOINT
    # ========================================================

    if (
        loop_index % SAVE_EVERY == 0
        or
        loop_index
        == len(remaining_df)
    ):

        # -------------------------------
        # ASR tokens
        # -------------------------------

        new_df = pd.DataFrame(
            all_new_rows
        )

        if len(old_asr_df) > 0:

            combined_asr = pd.concat(
                [
                    old_asr_df,
                    new_df,
                ],
                ignore_index=True
            )

        else:

            combined_asr = (
                new_df.copy()
            )

        if len(combined_asr) > 0:

            combined_asr = (
                combined_asr
                .drop_duplicates(
                    subset=[
                        "clip_id",
                        "asr_token_index",
                        "asr_start",
                        "asr_end",
                    ],
                    keep="last"
                )
                .sort_values(
                    [
                        "clip_id",
                        "asr_start",
                        "asr_token_index",
                    ]
                )
                .reset_index(
                    drop=True
                )
            )

            combined_asr.to_csv(
                FULL_ASR_CSV,
                index=False,
                encoding="utf-8-sig"
            )


        # -------------------------------
        # Completed clips
        # -------------------------------

        new_progress = pd.DataFrame(
            progress_rows
        )

        if ASR_PROGRESS_CSV.exists():

            existing_progress = (
                pd.read_csv(
                    ASR_PROGRESS_CSV
                )
            )

            progress_save = pd.concat(
                [
                    existing_progress,
                    new_progress,
                ],
                ignore_index=True
            )

        else:

            progress_save = (
                new_progress.copy()
            )

        if len(progress_save) > 0:

            progress_save = (
                progress_save
                .drop_duplicates(
                    subset=["clip_id"],
                    keep="last"
                )
            )

            progress_save.to_csv(
                ASR_PROGRESS_CSV,
                index=False,
                encoding="utf-8-sig"
            )


        # -------------------------------
        # Errors
        # -------------------------------

        if error_rows:

            error_save = (
                pd.DataFrame(
                    error_rows
                )
            )

            error_save.to_csv(
                ASR_ERROR_CSV,
                index=False,
                encoding="utf-8-sig"
            )


        print(
            f"\nCheckpoint "
            f"{loop_index}/"
            f"{len(remaining_df)}"
        )


# ============================================================
# RELOAD FINAL
# ============================================================

if FULL_ASR_CSV.exists():

    asr_full_df = pd.read_csv(
        FULL_ASR_CSV
    )

else:

    asr_full_df = pd.DataFrame()


if ASR_PROGRESS_CSV.exists():

    completed_final = pd.read_csv(
        ASR_PROGRESS_CSV
    )

else:

    completed_final = pd.DataFrame()


print()
print("=" * 90)
print("FULL ASR FINISHED")
print("=" * 90)

print(
    "Dataset sentences :",
    len(tokenized_df)
)

print(
    "Completed clips   :",
    (
        completed_final[
            "clip_id"
        ].nunique()
        if len(completed_final)
        else 0
    )
)

print(
    "ASR token rows    :",
    len(asr_full_df)
)

print(
    "Clips with tokens :",
    (
        asr_full_df[
            "clip_id"
        ].nunique()
        if len(asr_full_df)
        else 0
    )
)

print(
    "Errors            :",
    len(error_rows)
)

print("\nSaved:")
print(FULL_ASR_CSV)
print(ASR_PROGRESS_CSV)

if error_rows:
    print(ASR_ERROR_CSV)

In [ ]:
from pathlib import Path
import pandas as pd

OUTPUT_ROOT = Path(r"D:\Face_detection\dataset\word_level_v5\video001")

FULL_ASR_CSV = OUTPUT_ROOT / "asr_word_timestamps_full.csv"
ASR_PROGRESS_CSV = OUTPUT_ROOT / "asr_completed_clips.csv"
ASR_ERROR_CSV = OUTPUT_ROOT / "asr_errors.csv"

asr_full_df = pd.read_csv(FULL_ASR_CSV)
completed_df = pd.read_csv(ASR_PROGRESS_CSV)

print("Completed clips   :", completed_df["clip_id"].nunique())
print("ASR token rows    :", len(asr_full_df))
print("Clips with tokens :", asr_full_df["clip_id"].nunique())

if ASR_ERROR_CSV.exists():
    error_df = pd.read_csv(ASR_ERROR_CSV)
    print("Errors            :", len(error_df))
else:
    print("Errors            : 0")

In [ ]:
# ============================================================
# CELL 24 — FULL CHARACTER ALIGNMENT V3
# Align ทุกคำในทุกประโยค กับ ASR token ทั้งหมด
# ============================================================

import re
import unicodedata
import numpy as np
import pandas as pd
from difflib import SequenceMatcher


# ------------------------------------------------------------
# 1) Normalize text
# ------------------------------------------------------------

def normalize_for_alignment(text):
    text = unicodedata.normalize("NFC", str(text))
    text = re.sub(r"\s+", "", text)
    text = re.sub(
        r"[^\u0E00-\u0E7FA-Za-z0-9]",
        "",
        text
    )
    return text.lower().strip()


# ------------------------------------------------------------
# 2) Build ASR character map
# ------------------------------------------------------------

def build_asr_character_map(asr_tokens):

    asr_text = ""
    char_map = []

    for token_idx, token in enumerate(asr_tokens):

        raw_piece = str(token["asr_token"])
        piece = normalize_for_alignment(raw_piece)

        if not piece:
            continue

        start = float(token["asr_start"])
        end = float(token["asr_end"])

        probability = token.get(
            "asr_probability",
            np.nan
        )

        if pd.notna(probability):
            probability = float(probability)

        n = len(piece)
        duration = max(0.0, end - start)

        for j, ch in enumerate(piece):

            if duration > 0 and n > 0:
                ch_start = start + duration * (j / n)
                ch_end = start + duration * ((j + 1) / n)
            else:
                ch_start = start
                ch_end = end

            asr_text += ch

            char_map.append({
                "char": ch,
                "start": ch_start,
                "end": ch_end,
                "token_start": start,
                "token_end": end,
                "token_index": token_idx,
                "token_text": raw_piece,
                "probability": probability,
            })

    return asr_text, char_map


# ------------------------------------------------------------
# 3) Build target word spans
# ------------------------------------------------------------

def build_target_word_spans(words):

    spans = []
    current_pos = 0

    for original_index, word in words:

        norm_word = normalize_for_alignment(word)

        if not norm_word:
            continue

        start_pos = current_pos
        end_pos = current_pos + len(norm_word)

        spans.append({
            "word_index": int(original_index),
            "word": word,
            "norm_word": norm_word,
            "target_start": start_pos,
            "target_end": end_pos,
        })

        current_pos = end_pos

    target_text = "".join(
        x["norm_word"]
        for x in spans
    )

    return target_text, spans


# ------------------------------------------------------------
# 4) Character sequence alignment
# ------------------------------------------------------------

def create_character_alignment(
    target_text,
    asr_text
):

    matcher = SequenceMatcher(
        None,
        target_text,
        asr_text,
        autojunk=False
    )

    target_to_asr = {}

    for block in matcher.get_matching_blocks():

        for k in range(block.size):

            target_to_asr[
                block.a + k
            ] = block.b + k

    return target_to_asr, matcher.ratio()


# ------------------------------------------------------------
# 5) Align words
# ------------------------------------------------------------

def align_words_v3(
    target_words,
    asr_tokens
):

    target_text, word_spans = (
        build_target_word_spans(
            target_words
        )
    )

    asr_text, char_map = (
        build_asr_character_map(
            asr_tokens
        )
    )

    target_to_asr, sentence_similarity = (
        create_character_alignment(
            target_text,
            asr_text
        )
    )

    results = []

    for span in word_spans:

        start_pos = span["target_start"]
        end_pos = span["target_end"]

        matched_positions = []
        matched_chars = 0

        total_chars = max(
            1,
            end_pos - start_pos
        )

        for target_pos in range(
            start_pos,
            end_pos
        ):

            if target_pos in target_to_asr:

                asr_pos = target_to_asr[
                    target_pos
                ]

                if (
                    0 <= asr_pos
                    < len(char_map)
                ):
                    matched_positions.append(
                        asr_pos
                    )
                    matched_chars += 1

        coverage = (
            matched_chars
            / total_chars
        )

        # ไม่มี match
        if not matched_positions:

            results.append({
                "word_index": span["word_index"],
                "word": span["word"],
                "word_start": np.nan,
                "word_end": np.nan,
                "word_duration": np.nan,
                "char_coverage": 0.0,
                "alignment_confidence": 0.0,
                "sentence_similarity":
                    sentence_similarity,
                "matched_asr_text": "",
                "alignment_status": "review",
            })

            continue

        first = min(matched_positions)
        last = max(matched_positions)

        selected = (
            char_map[first:last + 1]
        )

        word_start = float(
            selected[0]["token_start"]
        )

        word_end = float(
            selected[-1]["token_end"]
        )

        duration = (
            word_end - word_start
        )

        matched_text = "".join(
            x["char"]
            for x in selected
        )

        probs = [
            x["probability"]
            for x in selected
            if pd.notna(
                x["probability"]
            )
        ]

        mean_prob = (
            float(np.mean(probs))
            if probs
            else np.nan
        )

        if pd.notna(mean_prob):

            confidence = (
                0.7 * coverage
                + 0.3 * mean_prob
            )

        else:

            confidence = coverage

        if coverage >= 0.80:
            status = "character_aligned"

        elif coverage >= 0.60:
            status = "partial_alignment"

        else:
            status = "review"

        results.append({
            "word_index": span["word_index"],
            "word": span["word"],
            "word_start": word_start,
            "word_end": word_end,
            "word_duration": duration,
            "char_coverage": coverage,
            "alignment_confidence":
                confidence,
            "sentence_similarity":
                sentence_similarity,
            "matched_asr_text":
                matched_text,
            "alignment_status":
                status,
        })

    return results


# ============================================================
# CHECK REQUIRED VARIABLES
# ============================================================

required_vars = [
    "word_occ_df",
    "asr_full_df",
    "OUTPUT_ROOT",
]

for name in required_vars:
    if name not in globals():
        raise RuntimeError(
            f"ไม่พบตัวแปร {name}"
        )


# ============================================================
# FULL ALIGNMENT
# ============================================================

aligned_full_rows = []

clips_with_asr = set(
    asr_full_df["clip_id"]
    .astype(str)
    .unique()
)

total_clips = (
    word_occ_df["clip_id"]
    .astype(str)
    .nunique()
)

processed_clips = 0


for clip_id, group in (
    word_occ_df.groupby("clip_id")
):

    clip_id = str(clip_id)

    if clip_id not in clips_with_asr:
        continue

    group = (
        group
        .sort_values("word_index")
        .reset_index(drop=True)
    )

    # ทุกคำใน sentence
    target_words = [
        (
            int(row["word_index"]),
            row["word"]
        )
        for _, row
        in group.iterrows()
    ]

    asr_group = (
        asr_full_df[
            asr_full_df[
                "clip_id"
            ].astype(str)
            == clip_id
        ]
        .sort_values(
            [
                "asr_start",
                "asr_token_index",
            ]
        )
        .reset_index(drop=True)
    )

    asr_tokens = (
        asr_group
        .to_dict("records")
    )

    results = align_words_v3(
        target_words,
        asr_tokens
    )

    original_map = {
        int(row["word_index"]):
            row.to_dict()
        for _, row
        in group.iterrows()
    }

    for result in results:

        idx = int(
            result["word_index"]
        )

        if idx not in original_map:
            continue

        original = (
            original_map[idx]
            .copy()
        )

        original.update(result)

        original[
            "alignment_source"
        ] = (
            "character_position_alignment_v3_full"
        )

        aligned_full_rows.append(
            original
        )

    processed_clips += 1

    if (
        processed_clips % 100 == 0
        or
        processed_clips == total_clips
    ):
        print(
            f"Processed "
            f"{processed_clips}/"
            f"{total_clips} clips"
        )


# ============================================================
# DATAFRAME
# ============================================================

alignment_full_df = pd.DataFrame(
    aligned_full_rows
)


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

FULL_ALIGNMENT_CSV = (
    OUTPUT_ROOT
    / "word_alignment_full_v3.csv"
)

alignment_full_df.to_csv(
    FULL_ALIGNMENT_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# SUMMARY
# ============================================================

print()
print("=" * 90)
print("FULL ALIGNMENT FINISHED")
print("=" * 90)

status_counts = (
    alignment_full_df[
        "alignment_status"
    ]
    .value_counts(
        dropna=False
    )
)

print(status_counts)

total_words = len(
    alignment_full_df
)

good = int(
    (
        alignment_full_df[
            "alignment_status"
        ]
        == "character_aligned"
    ).sum()
)

partial = int(
    (
        alignment_full_df[
            "alignment_status"
        ]
        == "partial_alignment"
    ).sum()
)

review = int(
    (
        alignment_full_df[
            "alignment_status"
        ]
        == "review"
    ).sum()
)

print()
print(
    "Total word occurrences :",
    total_words
)

print(
    "Character aligned      :",
    good
)

print(
    "Partial alignment      :",
    partial
)

print(
    "Review                 :",
    review
)

if total_words > 0:

    print(
        "Good rate             :",
        f"{good / total_words * 100:.2f}%"
    )

    print(
        "Good + Partial        :",
        f"{(good + partial) / total_words * 100:.2f}%"
    )


print()
print(
    "Mean confidence        :",
    round(
        alignment_full_df[
            "alignment_confidence"
        ].mean(),
        4
    )
)

print(
    "Mean char coverage     :",
    round(
        alignment_full_df[
            "char_coverage"
        ].mean(),
        4
    )
)

print()
print("Saved:")
print(FULL_ALIGNMENT_CSV)


# ============================================================
# PREVIEW
# ============================================================

display(
    alignment_full_df[
        [
            "clip_id",
            "split",
            "word_index",
            "word",

            "matched_asr_text",

            "word_start",
            "word_end",
            "word_duration",

            "char_coverage",
            "alignment_confidence",
            "sentence_similarity",

            "alignment_status",
        ]
    ].head(100)
)

In [ ]:
# ============================================================
# CELL 25 — BUILD FINAL WORD VOCABULARY
# เลือกคลาสคำจาก alignment ที่ผ่านจริง
# ============================================================

import pandas as pd
import numpy as np


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

MIN_TRAIN_COUNT = 10
MIN_VAL_COUNT = 2
MIN_TEST_COUNT = 2

MAX_VOCAB = 100

# ไม่ใช้ token ที่ไม่เหมาะเป็นคำเดี่ยว
EXCLUDED_WORDS = {
    "ๆ",
}


# ------------------------------------------------------------
# 1) ใช้เฉพาะ character_aligned
# ------------------------------------------------------------

aligned_good = alignment_full_df[
    alignment_full_df["alignment_status"]
    == "character_aligned"
].copy()


# ------------------------------------------------------------
# 2) Strict quality filter เพิ่มอีกชั้น
# ------------------------------------------------------------

aligned_good = aligned_good[
    (aligned_good["char_coverage"] >= 0.80)
    &
    (aligned_good["alignment_confidence"] >= 0.70)
    &
    (aligned_good["word_duration"] > 0.0)
    &
    (aligned_good["word_duration"] <= 2.50)
].copy()


# ------------------------------------------------------------
# 3) ตัดคำที่ไม่ต้องการ
# ------------------------------------------------------------

aligned_good["word"] = (
    aligned_good["word"]
    .astype(str)
    .str.strip()
)

aligned_good = aligned_good[
    ~aligned_good["word"].isin(
        EXCLUDED_WORDS
    )
].copy()


# ------------------------------------------------------------
# 4) Count ต่อ split
# ------------------------------------------------------------

word_split_counts = (
    aligned_good
    .groupby(
        ["word", "split"]
    )
    .size()
    .unstack(
        fill_value=0
    )
)


# ป้องกันกรณีบาง column ไม่มี
for split_name in [
    "train",
    "val",
    "test",
]:
    if split_name not in word_split_counts.columns:
        word_split_counts[split_name] = 0


word_split_counts = (
    word_split_counts[
        [
            "train",
            "val",
            "test",
        ]
    ]
)


# ------------------------------------------------------------
# Total
# ------------------------------------------------------------

word_split_counts[
    "total"
] = (
    word_split_counts[
        ["train", "val", "test"]
    ]
    .sum(axis=1)
)


# ------------------------------------------------------------
# 5) Eligible vocabulary
# ------------------------------------------------------------

eligible_vocab = (
    word_split_counts[
        (word_split_counts["train"] >= MIN_TRAIN_COUNT)
        &
        (word_split_counts["val"] >= MIN_VAL_COUNT)
        &
        (word_split_counts["test"] >= MIN_TEST_COUNT)
    ]
    .sort_values(
        [
            "train",
            "total",
        ],
        ascending=False
    )
    .head(MAX_VOCAB)
    .copy()
)


# ------------------------------------------------------------
# 6) class_id
# ------------------------------------------------------------

eligible_vocab = (
    eligible_vocab
    .reset_index()
)

eligible_vocab[
    "class_id"
] = np.arange(
    len(eligible_vocab)
)


# ------------------------------------------------------------
# 7) Save vocabulary
# ------------------------------------------------------------

VOCAB_CSV = (
    OUTPUT_ROOT
    / "word_vocabulary_v5.csv"
)

eligible_vocab.to_csv(
    VOCAB_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# SUMMARY
# ============================================================

print("=" * 85)
print("FINAL WORD VOCABULARY")
print("=" * 85)

print(
    "Aligned usable rows :",
    len(aligned_good)
)

print(
    "Unique usable words :",
    aligned_good["word"].nunique()
)

print(
    "Selected classes    :",
    len(eligible_vocab)
)

print()
print(
    "Criteria:"
)

print(
    "train >=",
    MIN_TRAIN_COUNT
)

print(
    "val   >=",
    MIN_VAL_COUNT
)

print(
    "test  >=",
    MIN_TEST_COUNT
)


print("\nSaved:")
print(VOCAB_CSV)


# ------------------------------------------------------------
# Display
# ------------------------------------------------------------

display(
    eligible_vocab[
        [
            "class_id",
            "word",
            "train",
            "val",
            "test",
            "total",
        ]
    ]
)

In [ ]:
# ============================================================
# CELL 26 — BUILD FINAL TRAIN MANIFEST + EXPORT ALL WORD CLIPS
# ============================================================

from pathlib import Path
import subprocess
import shutil
import numpy as np
import pandas as pd


# ------------------------------------------------------------
# SETTINGS
# ------------------------------------------------------------

LEFT_CONTEXT_SEC = 0.10
RIGHT_CONTEXT_SEC = 0.10


PROJECT_ROOT = Path(
    r"D:\Face_detection"
)

VIDEO_ID = "video001"

SOURCE_VIDEO = (
    PROJECT_ROOT
    / "dataset"
    / "raw_videos"
    / f"{VIDEO_ID}.webm"
)

FINAL_CLIP_ROOT = (
    PROJECT_ROOT
    / "dataset"
    / "word_level_v5"
    / VIDEO_ID
    / "clips_final"
)

FINAL_CLIP_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# FFmpeg check
# ------------------------------------------------------------

ffmpeg_path = shutil.which(
    "ffmpeg"
)

if ffmpeg_path is None:
    raise RuntimeError(
        "ไม่พบ ffmpeg"
    )

if not SOURCE_VIDEO.exists():
    raise FileNotFoundError(
        SOURCE_VIDEO
    )


# ============================================================
# 1) Vocabulary map
# ============================================================

vocab_map = dict(
    zip(
        eligible_vocab["word"],
        eligible_vocab["class_id"]
    )
)

selected_words = set(
    vocab_map.keys()
)


# ============================================================
# 2) เลือกเฉพาะตัวอย่างที่ผ่าน strict quality
# ============================================================

final_df = aligned_good[
    aligned_good["word"].isin(
        selected_words
    )
].copy()


# ------------------------------------------------------------
# class id
# ------------------------------------------------------------

final_df[
    "class_id"
] = final_df[
    "word"
].map(vocab_map)


# ------------------------------------------------------------
# เก็บ raw timestamp
# ------------------------------------------------------------

final_df[
    "raw_word_start"
] = final_df[
    "word_start"
]

final_df[
    "raw_word_end"
] = final_df[
    "word_end"
]


# ------------------------------------------------------------
# context
# ------------------------------------------------------------

final_df[
    "clip_start"
] = (
    final_df[
        "word_start"
    ]
    - LEFT_CONTEXT_SEC
)

final_df[
    "clip_end"
] = (
    final_df[
        "word_end"
    ]
    + RIGHT_CONTEXT_SEC
)


# ------------------------------------------------------------
# จำกัดไม่ให้ออกนอก sentence
# ------------------------------------------------------------

final_df[
    "clip_start"
] = np.maximum(
    final_df[
        "clip_start"
    ],
    final_df[
        "sentence_start"
    ]
)

final_df[
    "clip_end"
] = np.minimum(
    final_df[
        "clip_end"
    ],
    final_df[
        "sentence_end"
    ]
)


final_df[
    "clip_duration"
] = (
    final_df[
        "clip_end"
    ]
    - final_df[
        "clip_start"
    ]
)


# ------------------------------------------------------------
# unique word clip id
# ------------------------------------------------------------

final_df[
    "word_clip_id"
] = final_df.apply(
    lambda r:
    f"{r['clip_id']}_word{int(r['word_index']):03d}",
    axis=1
)


# ============================================================
# 3) Safe word folder
# ============================================================

def safe_word_name(word):

    word = str(word).strip()

    for ch in [
        '\\', '/', ':', '*',
        '?', '"', '<', '>', '|'
    ]:
        word = word.replace(
            ch,
            "_"
        )

    return word


# ============================================================
# 4) Export function
# ============================================================

def export_word_clip(
    input_video,
    output_path,
    start_time,
    end_time
):

    duration = (
        float(end_time)
        - float(start_time)
    )

    if duration <= 0:
        return False, "duration <= 0"

    output_path.parent.mkdir(
        parents=True,
        exist_ok=True
    )

    command = [
        ffmpeg_path,
        "-y",

        "-ss",
        f"{float(start_time):.3f}",

        "-i",
        str(input_video),

        "-t",
        f"{duration:.3f}",

        "-an",

        "-c:v",
        "libx264",

        "-preset",
        "fast",

        "-crf",
        "20",

        "-pix_fmt",
        "yuv420p",

        "-loglevel",
        "error",

        str(output_path),
    ]

    try:

        result = subprocess.run(
            command,
            stdout=subprocess.PIPE,
            stderr=subprocess.PIPE,
            text=True,
            timeout=60
        )

        if result.returncode != 0:
            return (
                False,
                result.stderr.strip()
            )

        if (
            not output_path.exists()
            or output_path.stat().st_size == 0
        ):
            return (
                False,
                "output missing/empty"
            )

        return True, ""

    except Exception as e:

        return (
            False,
            str(e)
        )


# ============================================================
# 5) EXPORT ALL
# ============================================================

final_df = (
    final_df
    .sort_values(
        [
            "split",
            "class_id",
            "clip_id",
            "word_index",
        ]
    )
    .reset_index(drop=True)
)


export_rows = []

success_count = 0
failed_count = 0

total = len(final_df)


print("=" * 85)
print("EXPORT FINAL WORD DATASET")
print("=" * 85)

print(
    "Total clips:",
    total
)

print(
    "Classes    :",
    final_df[
        "class_id"
    ].nunique()
)


for i, row in final_df.iterrows():

    split = str(
        row["split"]
    )

    word = str(
        row["word"]
    )

    class_id = int(
        row["class_id"]
    )

    folder_name = (
        f"{class_id:03d}_"
        f"{safe_word_name(word)}"
    )

    output_file = (
        FINAL_CLIP_ROOT
        / split
        / folder_name
        / (
            str(
                row["word_clip_id"]
            )
            + ".mp4"
        )
    )

    success, error = (
        export_word_clip(
            SOURCE_VIDEO,
            output_file,
            row["clip_start"],
            row["clip_end"]
        )
    )

    item = row.to_dict()

    if success:

        success_count += 1

        item[
            "export_status"
        ] = "success"

    else:

        failed_count += 1

        item[
            "export_status"
        ] = "failed"

    item[
        "export_error"
    ] = error

    item[
        "video_clip_path"
    ] = str(
        output_file
    )

    export_rows.append(
        item
    )


    if (
        (i + 1) % 100 == 0
        or
        i == total - 1
    ):

        print(
            f"[{i+1}/{total}] "
            f"success={success_count} "
            f"failed={failed_count}"
        )


# ============================================================
# 6) SAVE FINAL MANIFEST
# ============================================================

final_export_df = pd.DataFrame(
    export_rows
)


FINAL_MANIFEST = (
    OUTPUT_ROOT
    / "word_train_manifest_v5.csv"
)


final_export_df.to_csv(
    FINAL_MANIFEST,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 7) SUMMARY
# ============================================================

print()
print("=" * 85)
print("FINAL WORD DATASET FINISHED")
print("=" * 85)

print(
    "Total   :",
    len(final_export_df)
)

print(
    "Success :",
    success_count
)

print(
    "Failed  :",
    failed_count
)

if len(final_export_df):

    print(
        "Success rate:",
        f"{success_count / len(final_export_df) * 100:.2f}%"
    )


print(
    "\nClasses:",
    final_export_df[
        "word"
    ].nunique()
)


print("\nBy split:")

print(
    final_export_df[
        final_export_df[
            "export_status"
        ]
        == "success"
    ][
        "split"
    ].value_counts()
)


print("\nOutput:")
print(
    FINAL_CLIP_ROOT
)

print("\nManifest:")
print(
    FINAL_MANIFEST
)